# SOP OTDR observations — EllaLink / SSU-A

| | |
|---|---|
| **Version** | 1.0-public |
| **Status** | Public |
| **Author** | Miquel Masanas |
| **Project** | Submerse |
| **Cable system** | EllaLink (Sines, Portugal — Fortaleza, Brazil) |

---

Parameterized case study notebook. Select the event at the top of the setup cell:

- **Jan 20 2025** — Taiwan teleseismic event (OBS + land stations)
- **Apr 3 2025** — Reykjanes Ridge M6.9 (land stations only)

Pre-processing is described in `01_sop_otdr_preprocessing_intro.ipynb`.


## Setup and data loading


In [ ]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "soplib.py").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import scienceplots
import obspy
from scipy import signal
from datetime import time
from IPython import display

from src.case_studies import JAN2025_TAIWAN, APR2025_REYKJANES
from src.paths import DataPaths
from src.soplib import *

# --- Select case study -------------------------------------------------------
CASE = JAN2025_TAIWAN          # Taiwan teleseismic event (OBS + land stations)
# CASE = APR2025_REYKJANES     # Reykjanes M6.9 (land stations only)

PATHS = DataPaths.from_cwd(REPO_ROOT, case=CASE)

# --- Output options ----------------------------------------------------------
SAVE_FIGURES = True  # set True to write PNGs under FIGURES_DIR
FIGURES_DIR = str(PATHS.results_dir)


def save_figure(fig, filename: str, **kwargs) -> None:
    """Save *fig* when SAVE_FIGURES is enabled."""
    if not SAVE_FIGURES:
        return
    os.makedirs(FIGURES_DIR, exist_ok=True)
    path = os.path.join(FIGURES_DIR, filename)
    fig.savefig(
        path,
        dpi=kwargs.pop("dpi", 300),
        bbox_inches=kwargs.pop("bbox_inches", "tight"),
        **kwargs,
    )
    print(f"Saved {path}")


def normalize_obs(series):
    """Map series to [-1, 1] (OBS overlay style)."""
    return 2 * (((series - series.min()) / (series.max() - series.min())) - 0.5)


def normalize_01(series):
    """Map series to [0, 1]."""
    return (series - series.min()) / (series.max() - series.min())


os.makedirs(FIGURES_DIR, exist_ok=True)

%matplotlib inline
plt.style.use(['science', 'no-latex'])
plt.rcParams['figure.dpi']  = 100
plt.rcParams['savefig.dpi'] = 300

print('soplib loaded.')
print(f'Case: {CASE.label}')
print(f'Repo root: {REPO_ROOT}')
print(f'Save figures: {SAVE_FIGURES} -> {FIGURES_DIR}')


### Metadata visualization
Note that we are loading data that has been previously de-rotated for the seismic band observations

In [ ]:
print_hdf5_structure(str(PATHS.sop_hdf5_path(derotated=True)))

### Data loading - Fiber


In [ ]:
# -- Paths and case-specific settings -------------------------------------
PATHS = DataPaths.from_cwd(REPO_ROOT, case=CASE)
HDF5_DIR     = PATHS.hdf5_dir
CATALOG_PATH = PATHS.catalog_csv

START = CASE.day
END   = CASE.day
DAY   = CASE.day
MAG_THRESHOLD = CASE.mag_threshold

# -- Load de-rotated SOP data -----------------------------------------------
print(f'Loading SOP data {START} to {END} ...')
SOP_derotated = load_sops(HDF5_DIR, START, END, suffix="_derotated")
print(f'Loaded {len(SOP_derotated)} repeaters.')
print(f'Time range: {SOP_derotated[list(SOP_derotated.keys())[0]].index[[0,-1]]}')

# -- Event catalogue --------------------------------------------------------
catalogue = load_catalogue(str(CATALOG_PATH))
catalogue = catalogue[DAY:DAY]

print(f'Loaded {len(catalogue)} events.')
print(f'Median sample rate fs = {calculate_fs(SOP_derotated[0].index)} Hz.')

catalogue.head()


Standard deviation is computed using the un-filtered omega as it shows greater cross-correlation with the OBS envelope down the line, and kurtosis does the same for consistency.

In [ ]:
# ── 2.0 Compute derived quantities ────────────────────────────────────────
print('Computing derived quantities ...')

highcut = 0.1
ROLLING_WINDOW = '30s'
df_derived = {}

for key, df in SOP_derotated.items():
    S   = df[['S1', 'S2', 'S3']].dropna().to_numpy()
    dt  = get_dt(df)
    fs = 1/dt
    S_smooth = np.column_stack([
        lowpass_filter_sos(S[:, i], highcut=highcut, fs=fs, order=4)
        for i in range(3)
    ])
    
    # Geodesic angular rate
    omega_filt = sop_angular_rate_signed(S_smooth, dt)
    omega_s_filt =  bandpass_filter_sos(omega_filt, lowcut = 0.00666, highcut=highcut, fs=fs, order=4)
    omega_s_filt = pd.Series(omega_s_filt, index=df.index[:-1], name='omega_filt')
    omega = sop_angular_rate_signed(S, dt)
    omega_s = pd.Series(omega, index=df.index[:-1], name='omega')
    # Rolling statistics
    roll_std  = np.sqrt(omega_s.rolling(ROLLING_WINDOW,center=True).var())
    roll_kurt = omega_s.rolling(ROLLING_WINDOW, center=True).kurt()
    roll_std_s = np.sqrt(df[['S1','S2','S3']].rolling(ROLLING_WINDOW).var().sum(axis=1).to_numpy()[1::]) # we need to loose one sample as the angular rate is a differentiation and the first sample is undefined

    df_derived[key] = pd.DataFrame({
        'omega': omega_s,
        'omega_filtered': omega_s_filt,
        'speed_std': roll_std,
        'stokes_std': roll_std_s,
        'kurtosis': roll_kurt,
    })

print(f'Derived quantities computed for {len(df_derived)} repeaters.')
print(f'Columns: {list(df_derived[list(df_derived.keys())[0]].columns)}')

### Data visualization
#### Colorplots and selected repeater view


In [ ]:
plot_repeater_colormap(df_derived, quantity = 'speed_std',
                       title='std of omega',
                       cbar_label='',
                       cmap='viridis',
                       db_scale = False,
                       catalogue=catalogue, mag_min=MAG_THRESHOLD)


plt.show()


From the preliminary observation we can see some repeaters are permanently activated with high levels of angular rotation. We can isolate them to see wether they contain useful information or should be discarded to better visualize the rest.

Because localized time domain variations can be seen overall, it is worth keeping the unusually activated repeaters just in case the localized perturbations do correspond to a physically meaningful environment perturbation.  

Now we can do the same for some preliminary selected repeaters that display activation at around the earthquake of choice, which is the Taipes 6.0 mag earthquake at 16:17:26. For that we shorten the observed time span. It has several aftershocks I guess or 

In [ ]:
START = pd.Timestamp(CASE.event_window_start, tz="UTC")
END = pd.Timestamp(CASE.event_window_end, tz="UTC")

df_shorter = {
    ID: df_derived[ID].loc[START:END]
    for ID in df_derived.keys()
}

catalogue_shorter = catalogue.loc[START:END]
catalogue_shorter.head()


### Data loading - OBS


In [ ]:
# OBS probes (HDF5) — only available for some case studies

filtered_list_obs = []
data_list_obs = []

if CASE.has_obs:
    path = PATHS.obs_glob
    data_list_obs = load_all_OBS_h5_recursive(path)
    data_list_obs = [normalize_obs(ser) for ser in data_list_obs]

    for ser in data_list_obs:
        fs = calculate_fs(ser.index)
        vals = bandpass_filter_sos(ser.values, lowcut=0.001, highcut=0.1, fs=fs, order=4)
        ser_filtered = pd.Series(vals, index=ser.index, name=ser.name)
        filtered_list_obs.append(ser_filtered)

    for ser in data_list_obs:
        print(ser.name)

    display.Image(str(PATHS.obs_location_png))
else:
    print(f'No OBS data for case: {CASE.label}')


### Data loading - Land stations


In [ ]:
# Land broadband stations (MiniSEED via ObsPy)

mseed_folder = PATHS.land_mseed_dir
mseed_files = [
    mseed_folder / f for f in os.listdir(mseed_folder) if f.endswith(".mseed")
]

data_list_land = []
if mseed_files:
    for mseed_file in mseed_files:
        st = obspy.read(str(mseed_file))
        for tr in st:
            fs = tr.stats.sampling_rate
            startt = tr.stats.starttime.datetime
            idx = pd.date_range(
                start=pd.Timestamp(startt, tz="UTC"),
                periods=tr.stats.npts,
                freq=pd.Timedelta(seconds=1 / fs),
            )
            idx = pd.DatetimeIndex(idx).tz_convert("UTC")
            name_stripped = f"{tr.id}".split("(")[0].strip()
            ser = pd.Series(data=tr.data, index=idx, name=name_stripped)
            data_list_land.append(ser)
else:
    print("No .mseed files found in the folder.")

data_list_land = [normalize_obs(ser) for ser in data_list_land]

filtered_list_land = []
for ser in data_list_land:
    fs = 1.0 / ser.index.to_series().diff().median().total_seconds()
    ser_filtered = bandpass_filter_sos(
        ser.values, lowcut=0.00666, highcut=0.1, fs=fs, order=4
    )
    ser_filtered = pd.Series(ser_filtered, index=ser.index, name=ser.name)
    filtered_list_land.append(ser_filtered)

data_list = data_list_land + data_list_obs

display.Image(str(PATHS.land_stations_map_png))


Append the lists    

In [ ]:
# Combined filtered seismic traces (land first, then OBS — matches jan202024 ordering)
filtered_list = filtered_list_land + filtered_list_obs


In [ ]:

envelope_list = []
for ser in filtered_list:
    
    rolling_std = np.sqrt(ser.rolling(ROLLING_WINDOW,center=True).var())
    envelope_list.append(rolling_std)


In [ ]:



fig, axes = plt.subplots(figsize=[15, 8])
offset_scale = 1.0 

for i, (ser_filt, ser_env) in enumerate(zip(filtered_list, envelope_list)):
    y_off = i * offset_scale
    
    # Plot the Filtered signal (Blue)
    axes.plot(ser_filt.index, ser_filt.values + y_off, 
             linewidth=0.8, alpha=0.6)
    
    axes.text(ser_filt.index[-1], ser_filt.values[-1] + y_off, f"{ser_filt.name}",
             fontsize=9, color='black', fontweight='bold')
    
    axes.plot(ser_env.index, ser_env.values + y_off, 
             color='red', linewidth=1.2, alpha=0.9)

add_event_lines(axes, catalogue_shorter, mag_min=MAG_THRESHOLD)
axes.grid(True, alpha=0.2)
plt.show()



In this case there is plenty of activity registered even before 16:30, which to my understanding and assuming between 3.5 and 7 km/s is not expected to correspond with the earthquake in Taipei, some activity starting ast 17, in SUB05 and again at 18:10-18:20.   

Channel Z activity accross all SUBs match well at 17.10. Sub03 shows activity starting at around 17.40 in the XY plane.

## Spectral analysis

In here we show spectrograms of both the OBS and the fiber repeaters. As will be seen in the section after that, concerning the PSD analysis, the spectrograms for the fiber measurements are cut at 0.6 Hz to avoid a constantly populated band that limits the color dynamic range and difficoults observations.


In [ ]:
# Spectrograms — land stations, OBS (if any), and selected fiber repeaters
from datetime import time

selected_repeaters = [1, 3, 15, 20, 38, 59, 72, 73]
spec_path = FIGURES_DIR

# --- Land stations: sum components per station --------------------------------
if filtered_list_land:
    series_by_name = {ser.name: ser for ser in filtered_list_land}

    def station_base(name: str) -> str:
        return name.split(' (')[0].split('_')[0]

    stations = sorted({station_base(name) for name in series_by_name})
    summed_signals = {}
    for sta in stations:
        matching = [series_by_name[n] for n in series_by_name if station_base(n) == sta]
        if not matching:
            continue
        aligned = pd.concat(matching[:3], axis=1).dropna()
        summed = aligned.sum(axis=1)
        summed.name = f"{sta}_SUM"
        summed_signals[sta] = summed

    for sta, ser in summed_signals.items():
        spec_data, t_axis, f_axis = generateSpectrogram_scipy(
            ser.values,
            pd.Series(ser.index),
            windowSize=2**12,
            freqSpan=(0.0066, 0.1),
            overlapPercentage=0.9,
            nfft_size=2**15,
        )
        plot_spectrogram_results(
            10 * np.log10(spec_data),
            t_axis,
            f_axis,
            repeaterID=f"Land_{sta}",
            logplot=True,
            save=SAVE_FIGURES,
            path=spec_path,
            cmap='magma',
        )
        plt.tight_layout(rect=[0, 0, 1, 0.97])

# --- OBS stations: sum CH1/CH2/CHZ per SUB unit --------------------------------
if CASE.has_obs and filtered_list_obs:
    channels = ['CH1', 'CH2', 'CHZ']
    series_by_name = {ser.name: ser for ser in filtered_list_obs}
    stations = sorted({name.rsplit('_', 1)[0] for name in series_by_name})

    for sta in stations:
        try:
            ch_series = [series_by_name[f"{sta}_{ch}"] for ch in channels]
        except KeyError:
            print(f"Skipping OBS station {sta}: missing channel(s)")
            continue
        aligned = pd.concat(ch_series, axis=1).dropna()
        summed = aligned.sum(axis=1)
        summed.name = f"{sta}_SUM"

        spec_data, t_axis, f_axis = generateSpectrogram_scipy(
            summed.values,
            pd.Series(summed.index),
            windowSize=2**12,
            freqSpan=(0.0066, 0.1),
            overlapPercentage=0.9,
            nfft_size=2**15,
        )
        plot_spectrogram_results(
            10 * np.log10(spec_data),
            t_axis,
            f_axis,
            repeaterID=f"OBS_{sta}",
            logplot=True,
            save=SAVE_FIGURES,
            path=spec_path,
            cmap='magma',
        )
        plt.tight_layout(rect=[0, 0, 1, 0.97])

# --- Fiber: omega_filtered and S1+S2 for selected repeaters -------------------
for rep_name in selected_repeaters:
    if rep_name not in df_shorter:
        continue
    ser = normalize_obs(df_shorter[rep_name]['omega_filtered'])
    spec_data, t_axis, f_axis = generateSpectrogram_scipy(
        ser.values,
        ser.index,
        overlapPercentage=0.9,
        windowSize=2**9,
        freqSpan=(0.0066, 0.06),
    )
    plot_spectrogram_results(
        10 * np.log10(spec_data),
        t_axis,
        f_axis,
        repeaterID=f"Repeater_{rep_name} - Omega filtered",
        logplot=True,
        save=SAVE_FIGURES,
        path=spec_path,
        cmap='magma',
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])

for rep_name in selected_repeaters:
    if rep_name not in SOP_derotated:
        continue
    fs_repeater = calculate_fs(SOP_derotated[rep_name]['S1'].index)
    ser_repeater = bandpass_filter_sos(
        SOP_derotated[rep_name]['S1'] + SOP_derotated[rep_name]['S2'],
        0.00667,
        0.1,
        order=4,
        fs=fs_repeater,
    )
    ser_repeater = normalize_obs(ser_repeater)
    spec_data, t_axis, f_axis = generateSpectrogram_scipy(
        ser_repeater,
        SOP_derotated[rep_name]['S1'].index,
        overlapPercentage=0.9,
        windowSize=2**9,
        freqSpan=(0.0066, 0.06),
    )

    if CASE.event_window_start and CASE.event_window_end:
        t_start = pd.Timestamp(CASE.event_window_start).time()
        t_end = pd.Timestamp(CASE.event_window_end).time()
        t_mask = (t_axis.dt.time >= t_start) & (t_axis.dt.time <= t_end)
        spec_data_plot = spec_data[t_mask]
        t_axis_plot = t_axis[t_mask]
    else:
        spec_data_plot = spec_data
        t_axis_plot = t_axis

    plot_spectrogram_results(
        10 * np.log10(spec_data_plot),
        t_axis_plot,
        f_axis,
        repeaterID=f"Repeater_{rep_name} - S1 + S2 derotated and filtered",
        logplot=True,
        save=SAVE_FIGURES,
        path=spec_path,
        cmap='magma',
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])

plt.show()


We see major differences in the plot characteristics of both technologies. Let us see the PSDs to understand the frequency bands actually studied. In the current implementation, OBS signal has much larger lower-frequency response while the filtered omega signal has a much larger higher frequency response. In the OBS sthe frequency evolution within the events can be seen. In the filtered omega study, vertical frequency activations can be seen for several repeatres consistent with the approximate OBS perturbation times, at around 17:00, 17:20 to 17:30 and 17:40 as well as at 18:00.

In [ ]:
# Example PSD: repeater omega vs OBS station (Taiwan SUB03 when available)
fig, ax = plt.subplots(figsize=(14, 5))

rep_id = 38
ser_repeater = normalize_obs(df_derived[rep_id]['omega'])
fs_repeater = calculate_fs(ser_repeater.index)

f_rep_ns, p_rep_ns = psd_db(ser_repeater.values, fs_repeater, smooth_size=1)
ax.semilogx(
    f_rep_ns, p_rep_ns,
    label=f'Repeater {rep_id} (omega) raw',
    color='tab:blue', alpha=0.3, linewidth=1.2,
)

f_rep, p_rep = psd_db(ser_repeater.values, fs_repeater, smooth_size=80)
ax.semilogx(
    f_rep, p_rep,
    label=f'Repeater {rep_id} (omega) smoothed',
    color='tab:blue', alpha=1.0, linewidth=2.3,
)

if CASE.has_obs and data_list_obs:
    sub03 = [s for s in data_list_obs if 'SUB03' in s.name][:3]
    if len(sub03) == 3:
        ser_sub03_sum = sum(sub03)
        fs_sub03 = calculate_fs(ser_sub03_sum.index)
        f_sub_ns, p_sub_ns = psd_db(ser_sub03_sum.values, fs_sub03, smooth_size=1)
        ax.semilogx(
            f_sub_ns, p_sub_ns,
            label='SUB03 (CH1+CH2+CHZ) raw',
            color='tab:orange', alpha=0.3, linewidth=1.2,
        )
        f_sub, p_sub = psd_db(ser_sub03_sum.values, fs_sub03, smooth_size=80)
        ax.semilogx(
            f_sub, p_sub,
            label='SUB03 (CH1+CH2+CHZ) smoothed',
            color='tab:orange', alpha=1.0, linewidth=2.3,
        )
    else:
        print('SUB03 channels not found in OBS data; plotting fiber only.')
else:
    print('No OBS data for this case; plotting fiber PSD only.')

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (dB / Hz)")
ax.set_title(f"PSD — Repeater {rep_id} Omega" + (" and SUB03" if CASE.has_obs else ""))
ax.grid(True, which="both", ls="--", alpha=0.6)
ax.set_xlim([1e-3, 1.5])
ax.legend()
plt.tight_layout()
plt.show()

save_figure(fig, "OBS_vsOmega.png")


## Correlation analysis


From the previous visualization we can see how there is delay between the arrival times between the SUB units, as well as delay between the horizontal plane and the Z direction in a number of instances. Visually a lot of activity is concentrated in the  17:25 h range up to 18:30 hours for SUB03 and SUB06, with activity reaching 20:00 for SUB01, SUB05 and SUB07. Activity is also strong singe 16:00 h for SUB02 and SUB07.

Now we normalize, resample to the same rates and cross correlate with the SSU data, for a particular OBS channel.


**Best matches:** 



SUB01 - CH1 and CH2 have the fiber leading by around 83 mins h and we consider it non-related  
SUB01 - CHZ has the OBS leading by around 25 minutes,  with a second lag at around 50 mins  

SUB02 - CH1 and CH2 have the OBS leading by around 83 mins h and we consider it non-related  
SUB02 - CHZ have the OBS leading by 25 min,  with a second lag at around 50 mins  

SUB03 - CH1 and CH2: Have the least overall lag  
SUB03 - CHZ: OBS leads by around 25 minutes  

SUB04 - CH1 and CH2: Fiber leads by around 83 mins  
SUB04 - CHZ: OBS leads by around 25 minutes, with a second lag at around 50 mins  

SUB05 - CH1 and CH2: in CH1 OBS leads by around 20 mins, in CH2 OBS leads by around 30 mins  
SUB05 - CHZ: OBS leads by around 25 minutes, with a second lag at around 50 mins  

SUB06 - CH1 and CH2: in CH1 OBS leads by around 20 mins, in CH2 OBS leads by around 30 mins  
SUB06 - CHZ: OBS leads by around 25 minutes, with a second lag at around 50 mins  

SUB07 - CH1 and CH2: in CH1 OBS leads by around 75 mins  
SUB07 - CHZ: OBS leads by around 25 minutes, with a second lag at around 50 mins  


Readings of CHZ are very consistent, with fiber lag consistency for the horizontal plane but with observations lagging very little to nothing for SUB03, lags matching the CHZ one for SUB05 and SUB06, and possibly unrelated peaks when OBS leads by more than an hour.

Now we analyze quantitatively the correlation peak as a function of the repeater distribution, to find the fiber path that best corresponds to the OBS reading

In [ ]:
# Cross-correlation matrix: SOP speed_std vs seismic envelopes
result_dir = str(PATHS.results_dir)
os.makedirs(result_dir, exist_ok=True)


def to_utc_index(s: pd.Series) -> pd.Series:
    """Harmonize tz-naive vs UTC-aware indices before alignment."""
    if getattr(s.index, "tz", None) is None:
        return s.tz_localize("UTC")
    return s.tz_convert("UTC")


def peak_xcorr(sop: pd.Series, obs: pd.Series, max_lag_samples: int, dt_s: float):
    """Peak |xcorr| and lag (min) after UTC alignment and finite-sample masking."""
    sop_u = to_utc_index(sop.dropna())
    obs_u = to_utc_index(obs.dropna())
    obs_aligned = obs_u.reindex(sop_u.index, method="nearest")
    aligned = pd.concat([sop_u.rename("sop"), obs_aligned.rename("obs")], axis=1).dropna()
    if len(aligned) < 64:
        return np.nan, np.nan

    sop_n = (aligned["sop"] - aligned["sop"].mean()) / (aligned["sop"].std() + 1e-9)
    obs_n = (aligned["obs"] - aligned["obs"].mean()) / (aligned["obs"].std() + 1e-9)

    corr = signal.correlate(sop_n.values, obs_n.values, mode="full") / len(sop_n)
    lags = signal.correlation_lags(len(sop_n), len(obs_n))
    mask = (lags >= -max_lag_samples) & (lags <= max_lag_samples)
    lags_sec = lags[mask] * dt_s
    corr_win = corr[mask]
    if corr_win.size == 0:
        return np.nan, np.nan

    max_idx = int(np.argmax(np.abs(corr_win)))
    return float(np.abs(corr_win[max_idx])), float(lags_sec[max_idx] / 60.0)


r_indices = sorted(df_shorter.keys())
dt_sop = get_dt(df_shorter[1])
max_lag_s = 4 * 3600
max_lag_samples = int(max_lag_s / dt_sop)

# Matrix shape: (n_envelopes, n_repeaters) — rows = seismic envelopes, cols = fiber repeaters
peak_corr_mat = []
peak_lag_mat = []
obs_names = []

for idx, obs_raw in enumerate(envelope_list):
    peak_corr_row = []
    peak_lag_row = []
    denom = obs_raw.max() - obs_raw.min()
    if not np.isfinite(denom) or denom < 1e-12:
        obs_norm = obs_raw * 0.0
    else:
        obs_norm = (obs_raw - obs_raw.min()) / denom
    obs_names.append(
        obs_raw.name if hasattr(obs_raw, "name") and obs_raw.name is not None else str(idx)
    )

    for r in r_indices:
        sop_r = df_shorter[r]["speed_std"]
        peak, lag_min = peak_xcorr(sop_r, obs_norm, max_lag_samples, get_dt(sop_r))
        peak_corr_row.append(peak)
        peak_lag_row.append(lag_min)

    peak_corr_mat.append(peak_corr_row)
    peak_lag_mat.append(peak_lag_row)

peak_corr_mat = np.array(peak_corr_mat, dtype=float)
peak_lag_mat = np.array(peak_lag_mat, dtype=float)

print(
    f"peak_corr_mat: min={np.nanmin(peak_corr_mat):.3f}, "
    f"max={np.nanmax(peak_corr_mat):.3f}, "
    f"finite={np.isfinite(peak_corr_mat).sum()}/{peak_corr_mat.size}"
)
print("First envelopes:", obs_names[:3])


In [ ]:
xcorr_plot_threshold = 0.0

# --------- GENERAL SLICE SETUP HERE ---------
row_start = 1
row_end = -2            # negative: exclude last |row_end| rows (matches jan202024 notebook)
jump_step = 1
selected_slice = slice(row_start, row_end + 1, jump_step)

if row_end < 0:
    row_stop = peak_corr_mat.shape[0] + row_end + 1
else:
    row_stop = row_end + 1
row_indices = list(range(row_start, row_stop, jump_step))
num_rows = len(row_indices)
ytick_positions = np.arange(num_rows)
ytick_labels = [obs_names[idx] for idx in row_indices]

vmin = xcorr_plot_threshold
vmax = np.nanmax(peak_corr_mat)

import matplotlib.gridspec as gridspec

mean_peak_per_repeater = np.nanmean(peak_corr_mat[selected_slice, :], axis=0)
std_peak_per_repeater = np.nanstd(peak_corr_mat[selected_slice, :], axis=0)

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, width_ratios=[20, 1], height_ratios=[3, 1], hspace=0.4, wspace=0.09)

ax1 = fig.add_subplot(gs[0, 0])
im = ax1.imshow(
    peak_corr_mat[selected_slice, :],
    aspect="auto",
    interpolation="nearest",
    origin="lower",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
cax = fig.add_subplot(gs[0, 1])
cbar = fig.colorbar(im, cax=cax, format="%.2f")
cbar.set_label("Peak |Correlation|", fontsize=18)
cbar.ax.tick_params(labelsize=16)

ax1.set_xlabel("Repeater Index (Distance along Cable)", fontsize=20)
ax1.set_ylabel(f"Seismic Envelope (rows {row_start}-{row_end}, step={jump_step})", fontsize=20)
ax1.set_title(
    f"Peak Cross-Correlation |SOP-OBS| Rows {row_start}-{row_end} (step={jump_step}), "
    f"Color range: above {xcorr_plot_threshold}",
    fontsize=24,
)
ax1.set_yticks(ytick_positions)
ax1.set_yticklabels(ytick_labels, fontsize=16)
xtick_positions = np.linspace(0, len(r_indices) - 1, num=20, dtype=int)
ax1.set_xticks(xtick_positions)
ax1.set_xticklabels([str(r_indices[int(i)]) for i in xtick_positions], fontsize=16)
ax1.tick_params(axis="y", labelsize=16)
ax1.tick_params(axis="x", labelsize=16)

ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax2.bar(
    r_indices,
    mean_peak_per_repeater,
    color="tab:blue",
    width=1,
    align="center",
    yerr=std_peak_per_repeater,
    capsize=2,
    ecolor="black",
    error_kw={"elinewidth": 1.3},
)
ax2.set_ylabel("Mean Peak |Correlation|", fontsize=18)
ax2.set_xlabel("Repeater Index (Distance along Cable)", fontsize=18)
ax2.set_title(
    f"Mean Peak |Correlation| per Repeater (rows {row_start}-{row_end}, step={jump_step})",
    fontsize=18,
)
ax2.set_xlim(min(r_indices), max(r_indices))
ax2.set_ylim([xcorr_plot_threshold, 1])
ax2.set_xticks(xtick_positions)
ax2.set_xticklabels([str(r_indices[int(i)]) for i in xtick_positions], fontsize=14)
ax2.tick_params(axis="y", labelsize=14)
ax2.tick_params(axis="x", labelsize=14)

fig.delaxes(fig.add_subplot(gs[1, 1]))
plt.tight_layout()

save_figure(
    fig,
    f"peak_corr_mat_rows{row_start}-{row_end}_step{jump_step}_above_{xcorr_plot_threshold}.png",
)
plt.show()

# Lag heatmap (values in minutes — negative means OBS leads SOP)
lag_cmap = plt.get_cmap("coolwarm").copy()
lag_cmap.set_bad(color="white")

corr_mask = peak_corr_mat >= xcorr_plot_threshold
masked_peak_lag = np.ma.masked_where(~corr_mask, peak_lag_mat)

plt.figure(figsize=(14, 6))
im2 = plt.imshow(
    masked_peak_lag[selected_slice, :],
    aspect="auto",
    interpolation="nearest",
    origin="lower",
    cmap=lag_cmap,
)
plt.colorbar(
    im2,
    label="Lag at Peak Correlation (min)",
    fraction=0.046,
    pad=0.04,
    format="%.1f",
    ax=plt.gca(),
)
plt.xlabel("Repeater Index (Distance along Cable)", fontsize=20)
plt.ylabel(f"Seismic Envelope (rows {row_start}-{row_end}, step={jump_step})", fontsize=20)
plt.title(
    f"Lag at Peak Cross-Correlation (min) (Rows {row_start}-{row_end}, step={jump_step} "
    f"shown only where |xcorr| above {xcorr_plot_threshold})",
    fontsize=24,
)
plt.yticks(ticks=ytick_positions, labels=ytick_labels, fontsize=16)
plt.xticks(
    ticks=np.linspace(0, len(r_indices) - 1, num=10, dtype=int),
    labels=[str(r_indices[int(i)]) for i in np.linspace(0, len(r_indices) - 1, num=10, dtype=int)],
    fontsize=16,
)
cbar2 = plt.gcf().axes[-1]
cbar2.tick_params(labelsize=16)
cbar2.set_ylabel("Lag at Peak Correlation (min)", fontsize=18)
plt.tight_layout()
save_figure(
    plt.gcf(),
    f"peak_lag_mat_rows{row_start}-{row_end}_step{jump_step}_above_{xcorr_plot_threshold}.png",
)
plt.show()


In [ ]:
import scipy.signal as signal
import matplotlib.cm as cm
import os

def plot_cross_correlation_for_repeaters(repeater_ids, df_shorter, envelope_list, dt_sop, corr_threshold=0.0, max_lag_hr=4, output_dir_corr=None):
    """
    Plot cross-correlation of SOP repeater(s) vs all OBS envelopes.

    repeater_ids: list of SOP repeater numbers (keys in df_shorter)
    df_shorter: dict or DataFrame with repeater time series (expects ['speed_std'])
    envelope_list: list-like of OBS pd.Series (expects .name attribute for legend)
    dt_sop: float, sample spacing in seconds for SOP
    corr_threshold: float, correlation threshold to plot
    max_lag_hr: int, max lag in hours for window
    output_dir_corr: directory to save output pngs (optional)
    """
    if isinstance(repeater_ids, int):
        repeater_ids = [repeater_ids]
    if output_dir_corr is None:
        output_dir_corr = "correlation_plots"
    os.makedirs(output_dir_corr, exist_ok=True)
    obs_colors = cm.viridis(np.linspace(0, 1, len(envelope_list)))

    for rep in repeater_ids:
        sig = df_shorter[rep]['speed_std']
        max_lag_s = max_lag_hr * 3600
        max_lag_smpl = int(max_lag_s / dt_sop)
        plt.figure(figsize=(7, 3))
        for i, obs in enumerate(envelope_list):
            obs_norm = (obs - obs.min()) / (obs.max() - obs.min() + 1e-12)
            sop_u = to_utc_index(sig.dropna())
            obs_u = to_utc_index(obs_norm.dropna())
            obs_aligned = obs_u.reindex(sop_u.index, method='nearest')
            aligned = pd.concat(
                [sop_u.rename('sop'), obs_aligned.rename('obs')], axis=1
            ).dropna()
            if len(aligned) < 64:
                continue
            sig_n = (aligned['sop'] - aligned['sop'].mean()) / (aligned['sop'].std() + 1e-9)
            obs_n = (aligned['obs'] - aligned['obs'].mean()) / (aligned['obs'].std() + 1e-9)
            corr = signal.correlate(sig_n.values, obs_n.values, mode='full') / len(sig_n)
            corr *= np.sign(corr[np.abs(corr).argmax()])
            lags = signal.correlation_lags(len(sig_n), len(obs_n))
            mask = (lags >= -max_lag_smpl) & (lags <= max_lag_smpl)
            lags_min = lags[mask] * dt_sop / 60.0
            corr_win = corr[mask]
            if np.any(np.abs(corr_win) > corr_threshold):
                plt.plot(
                    lags_min,
                    corr_win,
                    color=obs_colors[i],
                    alpha=0.8,
                    lw=2,
                    label=str(getattr(obs, 'name', i)),
                )
        plt.axvline(0, color='black', lw=1, linestyle='-', label='Zero Lag')
        plt.axhline(0, color='grey', lw=0.8, alpha=0.5)
        plt.title(
            f"X-Correlation: Repeater {rep}   (Threshold: {corr_threshold})",
            fontsize=13,
        )
        plt.xlabel("Time Lag (min)", fontsize=12)
        plt.ylabel("Correlation Coefficient", fontsize=12)
        plt.legend(loc='upper right', fontsize=8, ncol=2)
        plt.grid(True, linestyle=':', alpha=0.4)
        plt.xlim(-max_lag_s / 60, max_lag_s / 60)
        plt.tight_layout()
        plt.minorticks_on()
        outname = os.path.join(output_dir_corr, f"cross_corr_REP_{rep}_all_OBS.png")
        save_figure(plt.gcf(), os.path.basename(outname))
        plt.show()




if CASE.id == "apr2025_reykjanes" : final_selected_list = [8, 18, 19, 29, 33, 55]
if CASE.id == "jan2025_taiwan" : final_selected_list = [1, 3, 15, 20, 38, 59, 72, 73]

plot_cross_correlation_for_repeaters(
    final_selected_list, df_shorter, envelope_list, dt_sop, output_dir_corr=FIGURES_DIR, corr_threshold=0.45
)

In [ ]:


events_filtered = filter_catalogue(catalogue.loc[START:END], MAG_THRESHOLD)

OBS_norms = []
if CASE.has_obs and filtered_list_obs:
    # Default: SUB03 channels (indices 6:8 in original Taiwan ordering)
    obs_indices = [
        i for i, ser in enumerate(filtered_list_obs)
        if ser.name.startswith('SUB03_')
    ][:3]
    if not obs_indices:
        obs_indices = list(range(min(3, len(filtered_list_obs))))

    for idx in obs_indices:
        obs = filtered_list_obs[idx]
        norm_obs = normalize_01(obs)
        norm_obs.name = getattr(obs, 'name', f'OBS {idx}')
        OBS_norms.append(norm_obs)
else:
    print('OBS overlay skipped (no OBS for this case).')

obs_colors_list = ['red', 'purple', 'green', 'brown', 'magenta', 'crimson']

for key in final_selected_list:
    if key not in df_shorter:
        continue
    fig, ax = plt.subplots(figsize=(7, 3))
    fig.suptitle(f'Geodesic Angular Rate {DAY} - Repeater {key} Comparison', fontsize=14)

    df = df_shorter[key]

    ax.plot(
        df.index, 2 * (normalize_01(df['omega']) - 0.5),
        linewidth=1.0, color='steelblue', alpha=0.6, label='Filtered Omega',
    )
    ax.plot(
        df.index, normalize_01(df['speed_std']),
        linewidth=1.0, color='orange', alpha=0.9, label='Speed STD',
    )

    for i, obs_norm in enumerate(OBS_norms):
        color = obs_colors_list[i % len(obs_colors_list)]
        offset = -3 + i * 0.7
        label = getattr(obs_norm, 'name', f'OBS {i}')
        ax.plot(
            obs_norm.index, obs_norm.values + offset,
            linewidth=1.2, color=color, alpha=0.9, label=label,
        )

    ax.set_ylabel(f'Rep {key}\n(Rel. Amp)', fontsize=9)
    add_event_lines(ax, catalogue_shorter, mag_min=MAG_THRESHOLD - 1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_ylim(-3, 1.05 + 0.7 * max(0, len(OBS_norms) - 1))
    ax.set_xlabel('Time (UTC)')
    ax.legend(loc='upper right', fontsize=8, ncol=4, frameon=True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_figure(fig, f"GeodesicRate_ObsOverlay_Rep{key}_{DAY}.png")
    plt.show()
    plt.close(fig)


## References

1. Zhan, Z., et al. (2021). Optical polarization-based seismic and water wave sensing on transoceanic cables. *Science*, 371(6530), 931–936. https://doi.org/10.1126/science.abe6648
2. Mecozzi, A. (2024). Polarization sensing using transponders in submarine optical cables. *Journal of Lightwave Technology*. (Verify DOI before citation.)
3. Costa, N., et al. (2023). HLLB-based distributed sensing on submarine cables. (Verify venue and DOI.)
4. Damask, J. N. (2005). *Polarization Optics in Telecommunications*. Springer.
5. Stokes parameters: https://en.wikipedia.org/wiki/Stokes_parameters

See also `references.bib` in the repository root.

